In [ ]:
# Root of the shared project folder on Google Drive.
# Team members mounted it at different places in their Drives, so all paths
# below are built from this single constant. Change it to match your setup.
DRIVE_ROOT = "/content/drive/MyDrive/furniture-retrieval"

In [ ]:
# SAM3 ve bağımlılıklarını kur
!pip install -q torch torchvision torchaudio
!pip install -q pycocotools timm einops psutil decord
!pip install -q pillow matplotlib
!pip uninstall numpy
print("✅ Kurulum tamamlandı!")

In [ ]:
# Google Drive'daki hammadde görsellerini bağla
from google.colab import drive
import os
drive.mount('/content/drive')
RAW_IMAGES_DIR = f'{DRIVE_ROOT}/raw-images'
IMAGE_PATH = os.path.join(RAW_IMAGES_DIR, '0.jpg')
os.environ['RAW_IMAGES_DIR'] = RAW_IMAGES_DIR
os.environ['IMAGE_PATH'] = IMAGE_PATH
if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"Beklenen görsel bulunamadı: {IMAGE_PATH}")
print(f"0.jpg için yol: {IMAGE_PATH}")

In [ ]:
import torch
print(f"GPU var mı? {torch.cuda.is_available()}")
print(f"GPU adı: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Yok'}")
print(f"GPU belleği: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# SAM3 için gerekli özel kütüphaneleri kontrol et
gerekli = ['pycocotools', 'timm', 'einops', 'psutil','decord']

for paket in gerekli:
    try:
        __import__(paket)
        print(f"✅ {paket} yüklü")
    except ImportError:
        print(f"❌ {paket} yüklü değil - yükleniyor...")
        !pip install -q {paket}
        print(f"✅ {paket} kuruldu")

In [ ]:
# Önce mevcut sam3 varsa temizle
!rm -rf sam3

# GitHub'dan klonla
!git clone https://github.com/facebookresearch/sam3.git

# Klonlama başarılı mı kontrol et
import os
if os.path.exists('sam3'):
    print("✅ SAM3 klasörü klonlandı")

    # sam3 klasörüne gir ve kur
    %cd sam3
    !pip install -e .

    print("\n=== Kurulum Kontrolü ===")
    # Kontrol et
    try:
        from sam3.model_builder import build_sam3_image_model
        from sam3.model.sam3_image_processor import Sam3Processor
        print("✅ SAM3 modülleri başarıyla import edildi!")
    except Exception as e:
        print(f"❌ Import hatası: {e}")
else:
    print("❌ Klonlama başarısız. Repo adı yanlış olabilir.")
    print("Alternatif olarak SAM2'yi deneyelim:")
    !git clone https://github.com/facebookresearch/sam2.git

In [ ]:

from google.colab import userdata
import os

# Token'ı Colab secrets'a manuel ekleyin:
# Sol tarafta 🔑 (Key) simgesine tıklayın
# "Add new secret" > Name: HF_TOKEN > Value: YOUR_TOKEN_HERE
# hf_REDACTED
# Sonra bu kodu çalıştırın:
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("✅ Hugging Face token yüklendi!")
except:
    print("❌ Token bulunamadı. Lütfen Colab secrets'a ekleyin.")

In [ ]:
import os
import json
import time
import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pycocotools import mask as mask_utils
#################################### For Image ####################################
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# Görsel yolu daha önce tanımlanmadıysa varsayılanı hazırla
RAW_IMAGES_DIR = os.environ.get('RAW_IMAGES_DIR', f'{DRIVE_ROOT}/raw-images')
IMAGE_PATH = os.environ.get('IMAGE_PATH', os.path.join(RAW_IMAGES_DIR, '0.jpg'))
if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"IMAGE_PATH tanımlı değil veya dosya bulunamadı: {IMAGE_PATH}")
print(f"Inference için kullanılacak dosya: {IMAGE_PATH}")

# GPU kontrolü ve ayarı
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Kullanılan cihaz: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    # GPU performansını artır
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

start_time = time.time()

# Load the model
print("Model yükleniyor...")
model_load_start = time.time()
model = build_sam3_image_model()
model = model.to(device)
model.eval()  # Inference modu
print(f"Model yükleme süresi: {time.time() - model_load_start:.2f} saniye")

processor = Sam3Processor(model)

# Load an image
print("Görüntü yükleniyor...")
image = Image.open(IMAGE_PATH).convert("RGB")

# Inference with no_grad for speed
print("Inference başlıyor...")
inference_start = time.time()

print("\n--- Inference öncesi GPU durumu ---")
!nvidia-smi

with torch.no_grad():  # Gradient hesaplama kapalı - daha hızlı
    inference_state = processor.set_image(image)
    output = processor.set_text_prompt(state=inference_state, prompt="Furnitures")

print("\n--- Inference sonrası GPU durumu ---")
!nvidia-smi

inference_time = time.time() - inference_start
print(f"Inference süresi: {inference_time:.2f} saniye")

# Get the masks, bounding boxes, and scores
masks, boxes, scores = output["masks"], output["boxes"], output["scores"]
print(f"Tespit edilen nesne sayısı: {len(masks) if masks is not None else 0}")

# Prepare COCO-style instance segmentation output
coco_output = {
    "images": [
        {
            "id": 1,
            "file_name": os.path.basename(IMAGE_PATH),
            "width": image.width,
            "height": image.height
        }
    ],
    "annotations": [],
    "categories": [
        {
            "id": 1,
            "name": "furniture",
            "supercategory": "furniture"
        }
    ]
}

if masks is not None and len(masks) > 0:
    annotation_id = 1
    for idx, mask in enumerate(masks):
        mask_np = mask.detach().cpu().numpy() if torch.is_tensor(mask) else mask
        if mask_np.ndim > 2:
            mask_np = mask_np.squeeze()
        mask_bin = (mask_np > 0).astype(np.uint8)
        if mask_bin.sum() == 0:
            continue
        rle = mask_utils.encode(np.asfortranarray(mask_bin))
        rle["counts"] = rle["counts"].decode("utf-8")
        bbox = mask_utils.toBbox(rle).tolist()
        area = float(mask_utils.area(rle))
        annotation = {
            "id": annotation_id,
            "image_id": 1,
            "category_id": 1,
            "segmentation": rle,
            "bbox": [float(x) for x in bbox],
            "area": area,
            "iscrowd": 0
        }
        if scores is not None and len(scores) > idx:
            score_val = scores[idx]
            score_val = score_val.item() if torch.is_tensor(score_val) else score_val
            annotation["score"] = float(score_val)
        coco_output["annotations"].append(annotation)
        annotation_id += 1

# Visualize the results
plt.figure(figsize=(12, 8))
plt.imshow(image)

# Show masks
if masks is not None and len(masks) > 0:
    for mask in masks:
        mask_np = mask.detach().cpu().numpy() if torch.is_tensor(mask) else mask
        if mask_np.ndim > 2:
            mask_np = mask_np.squeeze()
        color_mask = np.zeros((*mask_np.shape, 4))
        color = np.random.rand(3)
        color_mask[mask_np > 0] = [*color, 0.5]
        plt.imshow(color_mask)

# Show bounding boxes
if boxes is not None and len(boxes) > 0:
    for i, box in enumerate(boxes):
        box_np = box.detach().cpu().numpy() if torch.is_tensor(box) else box
        x1, y1, x2, y2 = box_np
        width = x2 - x1
        height = y2 - y1
        rect = plt.Rectangle((x1, y1), width, height, fill=False, edgecolor='red', linewidth=2)
        plt.gca().add_patch(rect)
        if scores is not None and len(scores) > i:
            score = scores[i].item() if torch.is_tensor(scores[i]) else scores[i]
            plt.text(x1, y1-5, f'{score:.2f}', color='red', fontsize=12,
                    bbox=dict(facecolor='white', alpha=0.7))

plt.axis('off')
plt.title('SAM3 Detection Results (0.jpg)')
plt.tight_layout()

output_image_path = os.path.join(os.path.dirname(IMAGE_PATH), 'sam3_0_output.png')
viz_start = time.time()
plt.savefig(output_image_path, dpi=150, bbox_inches='tight')
print(f"Görselleştirme süresi: {time.time() - viz_start:.2f} saniye")
print(f"Sonuç görseli: {output_image_path}")

coco_output_path = os.path.join(os.path.dirname(IMAGE_PATH), 'sam3_0_instances.json')
with open(coco_output_path, 'w', encoding='utf-8') as coco_file:
    json.dump(coco_output, coco_file, ensure_ascii=False, indent=2)
print(f"COCO instance segmentation çıktısı: {coco_output_path}")

total_time = time.time() - start_time
print(f"\n{'='*50}")
print(f"TOPLAM SÜRE: {total_time:.2f} saniye")
print(f"{'='*50}")

plt.show()